2.1 理论计算题
输入：3 × 32 × 32（通道数 × 高 × 宽）
卷积层：16 个卷积核，每个大小 3 × 5 × 5，填充 P=2，步幅 S=2

输出特征图尺寸
高/宽计算公式：
H_out = floor((H_in + 2P - K) / S) + 1
= floor((32 + 4 - 5) / 2) + 1
= floor(31 / 2) + 1 = 15 + 1 = 16
通道数 = 卷积核个数 = 16
因此输出特征图尺寸为 16 × 16 × 16。

单个输出通道的一个像素值所需的点乘次数
每个像素值对应卷积核与输入对应区域逐元素相乘后求和，乘法次数 = 卷积核元素个数 = 3 × 5 × 5 = 75。
因此需要 75 次点乘。

In [3]:
import torch
import torch.nn.functional as F

def max_pool2d(x, kernel_size, stride=None, padding=0):
    """
    x: (N, C, H, W)
    kernel_size: int or (int, int)
    stride: int or (int, int), default = kernel_size
    padding: int or (int, int), default 0
    """
    if isinstance(kernel_size, int):
        kH = kW = kernel_size
    else:
        kH, kW = kernel_size
    if stride is None:
        sH = sW = kH
    elif isinstance(stride, int):
        sH = sW = stride
    else:
        sH, sW = stride
    if isinstance(padding, int):
        pH = pW = padding
    else:
        pH, pW = padding

    N, C, H, W = x.shape
    # 填充
    x_pad = F.pad(x, (pW, pW, pH, pH), mode='constant', value=float('-inf'))
    H_pad, W_pad = H + 2*pH, W + 2*pW

    outH = (H_pad - kH) // sH + 1
    outW = (W_pad - kW) // sW + 1

    out = torch.zeros(N, C, outH, outW, device=x.device)
    for i in range(outH):
        for j in range(outW):
            h_start = i * sH
            h_end = h_start + kH
            w_start = j * sW
            w_end = w_start + kW
            window = x_pad[:, :, h_start:h_end, w_start:w_end]   # (N, C, kH, kW)
            # 使用 reshape 代替 view，避免非连续内存问题
            out[:, :, i, j] = window.reshape(N, C, -1).max(dim=2)[0]
    return out

# 测试
if __name__ == "__main__":
    x = torch.randn(2, 3, 8, 8)
    out_manual = max_pool2d(x, kernel_size=3, stride=2, padding=1)
    out_torch = F.max_pool2d(x, kernel_size=3, stride=2, padding=1)
    print(f"Input shape: {x.shape}")
    print(f"Output shape (kernel=3, stride=2, pad=1): {out_manual.shape}")
    print(f"Output (first batch, first channel):\n{out_manual[0,0,:4,:4]}")
    assert torch.allclose(out_manual, out_torch, atol=1e-6), "Mismatch with PyTorch MaxPool2d"
    print("Test passed: manual max pooling matches PyTorch result.")

Input shape: torch.Size([2, 3, 8, 8])
Output shape (kernel=3, stride=2, pad=1): torch.Size([2, 3, 4, 4])
Output (first batch, first channel):
tensor([[0.6556, 1.0657, 1.0657, 0.6004],
        [0.4408, 1.0657, 1.3933, 1.3933],
        [0.4715, 0.4715, 1.7684, 1.3933],
        [1.3903, 1.3903, 1.7684, 1.0016]])
Test passed: manual max pooling matches PyTorch result.


3.1 理论计算题
假设输入和输出通道数均为 C，无偏置。

一个 5×5 卷积层的参数量
C × C × 5 × 5 = 25 C²

两个串联的 3×3 卷积层（通道数均为 C）的总参数量
第一层：C × C × 3 × 3 = 9 C²
第二层：C × C × 3 × 3 = 9 C²
总计：18 C²

因此，两个 3×3 卷积的参数比一个 5×5 卷积更少（18C² < 25C²），同时增加了非线性表达能力。

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)

# 测试
if __name__ == "__main__":
    block = NiNBlock(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
    x = torch.randn(2, 3, 28, 28)
    out = block(x)
    print(f"NiN block forward output shape: {out.shape}")

NiN block forward output shape: torch.Size([2, 64, 28, 28])


4.1 理论计算题：Batch Normalization
给定小批量中某一通道某空间位置的四个样本值：x1=2, x2=4, x3=6, x4=8。
参数：γ = 2, β = 1, ε = 0。

计算步骤：

均值 μ = (2+4+6+8)/4 = 5

方差 σ² = [(2-5)²+(4-5)²+(6-5)²+(8-5)²]/4 = (9+1+1+9)/4 = 5

标准化：x̂_i = (x_i - μ) / √(σ²+ε) = (x_i - 5)/√5

变换：y_i = γ·x̂_i + β = 2·(x_i - 5)/√5 + 1

计算结果：
y1 = 2·(-3)/√5 + 1 = -6/√5 + 1 ≈ -1.683
y2 = 2·(-1)/√5 + 1 = -2/√5 + 1 ≈ 0.106
y3 = 2·(1)/√5 + 1 = 2/√5 + 1 ≈ 1.894
y4 = 2·(3)/√5 + 1 = 6/√5 + 1 ≈ 3.683

In [5]:
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            if in_channels == out_channels and stride == 1:
                self.shortcut = nn.Identity()
            else:
                raise ValueError("Need use_1x1conv=True when channels or stride changes")

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        shortcut = self.shortcut(x)
        out += shortcut
        return F.relu(out)

# 测试
if __name__ == "__main__":
    block = Residual(32, 64, use_1x1conv=True, stride=2)
    x = torch.randn(2, 32, 56, 56)
    out = block(x)
    print(f"Residual block output shape: {out.shape}")

Residual block output shape: torch.Size([2, 64, 28, 28])


5.1 理论计算题：微调
为什么底层特征提取层设置较小学习率（或冻结），而顶层输出层设置较大学习率？

底层特征通常是通用的（如边缘、纹理），这些特征在源数据集上已经学习得很好，若学习率过大容易破坏这些通用特征，导致过拟合或灾难性遗忘。

顶层输出层随机初始化，需要从目标数据集中快速学习特定分类（或回归）任务，因此需要较大学习率来加速收敛。

目标数据集非常小且与源数据集非常相似时，应采取什么策略防止过拟合？

冻结大部分底层网络，只微调顶层几层（或仅全连接层）。

使用更小的学习率、更强的正则化（如权重衰减、Dropout）。

采用数据增强，扩充目标数据集。

早停策略，根据验证集性能提前终止训练。

In [6]:
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

print("transforms:")
print(augmentation_pipeline)

# 使用示例：
# from PIL import Image
# img = Image.open('example.jpg')
# tensor_img = augmentation_pipeline(img)

transforms:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


6.1 理论计算题：交并比 (IoU)
真实框 A = [10,10,50,50]，预测框 B = [30,30,70,70]。

计算步骤：

交集左上角：(max(10,30), max(10,30)) = (30,30)

交集右下角：(min(50,70), min(50,70)) = (50,50)

交集宽高：20, 20，面积 = 20 × 20 = 400

A 面积：(50-10)² = 1600，B 面积：1600

并集面积：1600 + 1600 - 400 = 2800

IoU = 400 / 2800 = 1/7 ≈ 0.142857

In [7]:
def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    logits: (N, C) 未归一化的分数
    targets: (N,) 真实类别索引 (0 <= targets < C)
    epsilon: 平滑因子
    """
    C = logits.size(1)
    smooth_targets = torch.full_like(logits, epsilon / (C - 1))
    smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)
    log_probs = F.log_softmax(logits, dim=1)
    loss = -(smooth_targets * log_probs).sum(dim=1).mean()
    return loss

# 测试
if __name__ == "__main__":
    logits = torch.randn(4, 10, requires_grad=True)
    targets = torch.tensor([0, 1, 2, 3])
    loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)
    print(f"标签平滑交叉熵损失值: {loss}")

标签平滑交叉熵损失值: 2.6657845973968506
